In [2]:
import pandas as pd

df_game_team_stats = pd.read_csv(r"C:\Users\zacle\Desktop\Serene\Project\NHL\ETL\extract\raw_data\game_teams_stats.csv")
df_game_team_stats.head(5)
df_game_team_stats.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 52610 entries, 0 to 52609
Data columns (total 17 columns):
 #   Column                  Non-Null Count  Dtype  
---  ------                  --------------  -----  
 0   game_id                 52610 non-null  int64  
 1   team_id                 52610 non-null  int64  
 2   HoA                     52610 non-null  object 
 3   won                     52610 non-null  bool   
 4   settled_in              52610 non-null  object 
 5   head_coach              52582 non-null  object 
 6   goals                   52602 non-null  float64
 7   shots                   52602 non-null  float64
 8   hits                    47682 non-null  float64
 9   pim                     52602 non-null  float64
 10  powerPlayOpportunities  52602 non-null  float64
 11  powerPlayGoals          52602 non-null  float64
 12  faceOffWinPercentage    30462 non-null  float64
 13  giveaways               47682 non-null  float64
 14  takeaways               47682 non-null

In [4]:
# Create a new dataframe from df_game_team_stats (copying original to preserve data)
df_clean_game_team_stats = df_game_team_stats.copy()

In [6]:
df_clean_game_team_stats = df_clean_game_team_stats.rename(columns={'HoA': 'hoa'})
df_clean_game_team_stats.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 52610 entries, 0 to 52609
Data columns (total 17 columns):
 #   Column                  Non-Null Count  Dtype  
---  ------                  --------------  -----  
 0   game_id                 52610 non-null  int64  
 1   team_id                 52610 non-null  int64  
 2   hoa                     52610 non-null  object 
 3   won                     52610 non-null  bool   
 4   settled_in              52610 non-null  object 
 5   head_coach              52582 non-null  object 
 6   goals                   52602 non-null  float64
 7   shots                   52602 non-null  float64
 8   hits                    47682 non-null  float64
 9   pim                     52602 non-null  float64
 10  powerPlayOpportunities  52602 non-null  float64
 11  powerPlayGoals          52602 non-null  float64
 12  faceOffWinPercentage    30462 non-null  float64
 13  giveaways               47682 non-null  float64
 14  takeaways               47682 non-null

In [8]:
#Rename columns for consistency
#Function to add an underscore before uppercase letters and convert to lowercase
import re

def rename_columns(col_name):
    return re.sub(r'([a-z0-9])([A-Z])', r'\1_\2', col_name).lower()

# Apply the function to all column names
df_clean_game_team_stats.columns = [rename_columns(col) for col in df_clean_game_team_stats.columns]
df_clean_game_team_stats.info()

# To ensure data consistency with df - game_goalie_stats and game_skater_stats, rename pim column
df_clean_game_team_stats.rename(columns={'pim': 'penalty_minutes'}, inplace=True)

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 52610 entries, 0 to 52609
Data columns (total 17 columns):
 #   Column                    Non-Null Count  Dtype  
---  ------                    --------------  -----  
 0   game_id                   52610 non-null  int64  
 1   team_id                   52610 non-null  int64  
 2   hoa                       52610 non-null  object 
 3   won                       52610 non-null  bool   
 4   settled_in                52610 non-null  object 
 5   head_coach                52582 non-null  object 
 6   goals                     52602 non-null  float64
 7   shots                     52602 non-null  float64
 8   hits                      47682 non-null  float64
 9   pim                       52602 non-null  float64
 10  power_play_opportunities  52602 non-null  float64
 11  power_play_goals          52602 non-null  float64
 12  face_off_win_percentage   30462 non-null  float64
 13  giveaways                 47682 non-null  float64
 14  takeaw

In [10]:
# Null - All - check for missing or null values entire dataset
df_clean_game_team_stats.isnull().sum()

game_id                         0
team_id                         0
hoa                             0
won                             0
settled_in                      0
head_coach                     28
goals                           8
shots                           8
hits                         4928
penalty_minutes                 8
power_play_opportunities        8
power_play_goals                8
face_off_win_percentage     22148
giveaways                    4928
takeaways                    4928
blocked                      4928
start_rink_side              2392
dtype: int64

In [12]:
# Remove null values from 'head coach' since the number is trivial
df_clean_game_team_stats = df_clean_game_team_stats.dropna(subset=['head_coach'])
df_clean_game_team_stats.isnull().sum()

# Fill null values in the column 'start_rink_side' with 'unknown'
df_clean_game_team_stats['start_rink_side'] = df_clean_game_team_stats['start_rink_side'].fillna('unknown')

In [14]:
df_clean_game_team_stats.isnull().sum()

game_id                         0
team_id                         0
hoa                             0
won                             0
settled_in                      0
head_coach                      0
goals                           0
shots                           0
hits                         4920
penalty_minutes                 0
power_play_opportunities        0
power_play_goals                0
face_off_win_percentage     22140
giveaways                    4920
takeaways                    4920
blocked                      4920
start_rink_side                 0
dtype: int64

In [16]:
# duplicates : 28875 
# Count unique 'game_id' values
unique_ids = df_clean_game_team_stats['game_id'].nunique()

# Count total number of rows
total_rows = len(df_clean_game_team_stats)

# Display the results
print(f"Unique game_ids: {unique_ids}")
print(f"Total rows: {total_rows}")
print(f"There are {total_rows - unique_ids} rows with duplicates.")

Unique game_ids: 23721
Total rows: 52582
There are 28861 rows with duplicates.


In [18]:
# 5140 duplicate rows
# Additional Duplicate Check for 'game_id', 'team_id' combinations
duplicates_id = df_clean_game_team_stats[df_clean_game_team_stats.duplicated(subset=['game_id', 'team_id'], keep=False)]

# Sort rows with duplicated results in ascending order by 'game_id', 'team_id'
duplicates_id_sorted = duplicates_id.sort_values(by=['game_id', 'team_id'], ascending=True)

# Count total number of rows with duplicates based on 'game_id', 'team_id'
total_duplicates = df_clean_game_team_stats.duplicated(subset=['game_id', 'team_id'], keep=False).sum()

# Count the unique combinations of 'game_id', 'team_id'
unique_combinations = df_clean_game_team_stats[['game_id', 'team_id']].drop_duplicates().shape[0]

# Count total number of rows
total_rows = len(df_clean_game_team_stats)

# Display the results
print(f"Unique 'game_id', 'team_id' combinations: {unique_combinations}")
print(f"Total rows: {total_rows}")
print(f"There are {total_rows - unique_combinations} rows with duplicates based on 'game_id', 'team_id'.")

# Display the sorted duplicates for further review
print("Sorted Duplicates based on 'game_id', 'team_id':")
print(duplicates_id_sorted[['game_id', 'team_id']])
duplicates_id_sorted.head(5)

Unique 'game_id', 'team_id' combinations: 47442
Total rows: 52582
There are 5140 rows with duplicates based on 'game_id', 'team_id'.
Sorted Duplicates based on 'game_id', 'team_id':
          game_id  team_id
47170  2018020001        8
47178  2018020001        8
47171  2018020001       10
47179  2018020001       10
47172  2018020002        6
...           ...      ...
45396  2019040652       90
45392  2019040653       87
45398  2019040653       87
45393  2019040653       90
45399  2019040653       90

[10280 rows x 2 columns]


,game_id,team_id,hoa,won,settled_in,head_coach,goals,shots,hits,penalty_minutes,power_play_opportunities,power_play_goals,face_off_win_percentage,giveaways,takeaways,blocked,start_rink_side
47170,2018020001,8,away,False,OT,Claude Julien,2.0,36.0,34.0,6.0,4.0,1.0,41.3,10.0,5.0,16.0,left
47178,2018020001,8,away,False,OT,Claude Julien,2.0,36.0,34.0,6.0,4.0,1.0,41.3,10.0,5.0,16.0,left
47171,2018020001,10,home,True,OT,Mike Babcock,3.0,26.0,18.0,8.0,3.0,1.0,58.7,21.0,5.0,24.0,left
47179,2018020001,10,home,True,OT,Mike Babcock,3.0,26.0,18.0,8.0,3.0,1.0,58.7,21.0,5.0,24.0,left
47172,2018020002,6,away,False,REG,Bruce Cassidy,0.0,25.0,28.0,32.0,2.0,0.0,68.3,3.0,5.0,12.0,left


In [20]:
# Upon visual inspection of the 1st 30 rows shows that there are 5140 rows with true duplicates. 
# Remove duplicates based on 'game_id', 'team_id', 'goals'(just to to be sure!)
df_clean_game_team_stats = df_clean_game_team_stats.drop_duplicates(subset=['game_id', 'team_id', 'goals'], keep='first')

# Display the cleaned dataframe (first 30 rows as an example)
df_clean_game_team_stats.head(10)
df_clean_game_team_stats.info()

<class 'pandas.core.frame.DataFrame'>
Index: 47442 entries, 0 to 52607
Data columns (total 17 columns):
 #   Column                    Non-Null Count  Dtype  
---  ------                    --------------  -----  
 0   game_id                   47442 non-null  int64  
 1   team_id                   47442 non-null  int64  
 2   hoa                       47442 non-null  object 
 3   won                       47442 non-null  bool   
 4   settled_in                47442 non-null  object 
 5   head_coach                47442 non-null  object 
 6   goals                     47442 non-null  float64
 7   shots                     47442 non-null  float64
 8   hits                      42522 non-null  float64
 9   penalty_minutes           47442 non-null  float64
 10  power_play_opportunities  47442 non-null  float64
 11  power_play_goals          47442 non-null  float64
 12  face_off_win_percentage   25302 non-null  float64
 13  giveaways                 42522 non-null  float64
 14  takeaways  

In [22]:
# Check again for duplicates. No duplicates. Success!
# Additional Duplicate Check for 'game_id', 'team_id' combinations
duplicates_id = df_clean_game_team_stats[df_clean_game_team_stats.duplicated(subset=['game_id', 'team_id'], keep=False)]

# Sort rows with duplicated results in ascending order by 'game_id', 'team_id'
duplicates_id_sorted = duplicates_id.sort_values(by=['game_id', 'team_id'], ascending=True)

# Count total number of rows with duplicates based on 'game_id', 'team_id'
total_duplicates = df_clean_game_team_stats.duplicated(subset=['game_id', 'team_id'], keep=False).sum()

# Count the unique combinations of 'game_id', 'team_id'
unique_combinations = df_clean_game_team_stats[['game_id', 'team_id']].drop_duplicates().shape[0]

# Count total number of rows
total_rows = len(df_clean_game_team_stats)

# Display the results
print(f"Unique 'game_id', 'team_id' combinations: {unique_combinations}")
print(f"Total rows: {total_rows}")
print(f"There are {total_rows - unique_combinations} rows with duplicates based on 'game_id', 'team_id'.")

# Display the sorted duplicates for further review
print("Sorted Duplicates based on 'game_id', 'team_id':")
print(duplicates_id_sorted[['game_id', 'team_id']])
duplicates_id_sorted.head(30)

Unique 'game_id', 'team_id' combinations: 47442
Total rows: 47442
There are 0 rows with duplicates based on 'game_id', 'team_id'.
Sorted Duplicates based on 'game_id', 'team_id':
Empty DataFrame
Columns: [game_id, team_id]
Index: []


,game_id,team_id,hoa,won,settled_in,head_coach,goals,shots,hits,penalty_minutes,power_play_opportunities,power_play_goals,face_off_win_percentage,giveaways,takeaways,blocked,start_rink_side


In [24]:
#Syntax - Rename specific column : df = df.rename(columns={'old_column_name': 'new_column_name'})
df_clean_game_team_stats = df_clean_game_team_stats.rename(columns={'pim':'penalty_mins'})
df_clean_game_team_stats = df_clean_game_team_stats.rename(columns={'power_play_opportunities':'power_play_opps'})

In [26]:
# Convert DataType to string
df_clean_game_team_stats['game_id'] = df_clean_game_team_stats['game_id'].astype(str)
df_clean_game_team_stats['team_id'] = df_clean_game_team_stats['team_id'].astype(str)
df_clean_game_team_stats['won'] = df_clean_game_team_stats['won'].astype(str)
df_clean_game_team_stats['goals'] = df_clean_game_team_stats['goals'].astype(int)
df_clean_game_team_stats['shots'] = df_clean_game_team_stats['shots'].astype(int)
df_clean_game_team_stats['penalty_minutes'] = df_clean_game_team_stats['penalty_minutes'].astype(int)
df_clean_game_team_stats['power_play_opps'] = df_clean_game_team_stats['power_play_opps'].astype(int)
df_clean_game_team_stats['power_play_goals'] = df_clean_game_team_stats['power_play_goals'].astype(int)

In [34]:
df_clean_game_team_stats.dtypes

game_id                     object
team_id                     object
hoa                         object
won                         object
settled_in                  object
head_coach                  object
goals                        int32
shots                        int32
hits                       float64
penalty_minutes              int32
power_play_opps              int32
power_play_goals             int32
face_off_win_percentage    float64
giveaways                  float64
takeaways                  float64
blocked                    float64
start_rink_side             object
dtype: object

In [38]:
#save file locally
df_clean_game_team_stats.to_csv(r"C:\Users\zacle\Desktop\Serene\Project\NHL\ETL\clean\game_team_stats.csv", index=False)